# Node 00 · V3 起始文件生成 (`all_candidates_v3_start.csv`)

**目的**:从权威并集 `final_candidates_paper.csv` 提取 V3 干净出发文件 —— 只保留
身份 + 出身 + ADMET/QM 属性,剥离一切 docking / 选择标志 / 复合奖励 / rl_ 列。

**留痕原则**:每一步都打印证据、写 manifest、`DRY_RUN` 门控;算法在
`core/Node00_property_carryforward.py`,notebook 只负责编排与留痕(不内联算法)。

**边界**:本步只固定身份 *范围*;权威分子 *对象*(质子化/电荷微观态、3D 主结构)
在 Node 01–02 建立,不从这里的列推断。`qm_status` 标出 24 个中性化 QM 需重算。


## 0 · 配置与安全开关

In [11]:
# ── 全局配置与安全开关 ──────────────────────────────────
from pathlib import Path
import os, sys, json, hashlib, platform, datetime
# 安全默认：不做任何写/构建/运行，只读检查
DRY_RUN = False
# 根目录：默认取 notebook 所在目录；HPC 上应为 /mnt/hpc/home/wanghongmei/lang2
PROJECT_ROOT = Path(os.environ.get("PRISM_ROOT", "..")).resolve()
CORE_DIR    = PROJECT_ROOT / "core"
RESULTS_DIR = PROJECT_ROOT / "results"
# results/ 下的阶段节点（保持既定框架，不新增根节点）
STAGE_DIRS = [
    "00_scope", "01_microstates", "02_master_molecules", "03_pdbqt",
    "04_docking", "05_broad_spectrum_ranking", "06_md_selection",
    "07_parameterization", "08_md", "09_qc",
]
def utc_now():
    return datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
def file_sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()
print(f"DRY_RUN       = {DRY_RUN}")
print(f"PROJECT_ROOT  = {PROJECT_ROOT}")
print(f"CORE_DIR      = {CORE_DIR}")
print(f"RESULTS_DIR   = {RESULTS_DIR}")
print(f"时间(UTC)     = {utc_now()}")

# ── 本步(Node 00)专用路径 —— 基于上面的框架派生 ──────────────
SCOPE_DIR    = RESULTS_DIR / "00_scope"
SRC_CSV      = RESULTS_DIR / "final_candidates_paper.csv"   # 权威并集(输入)
OUT_CSV      = SCOPE_DIR / "all_candidates_v3_start.csv"
OUT_MANIFEST = SCOPE_DIR / "all_candidates_v3_start.manifest.json"
OUT_TRACE    = SCOPE_DIR / "all_candidates_v3_start.trace.json"
print(f"SRC_CSV       = {SRC_CSV}")
print(f"OUT_CSV       = {OUT_CSV}")

DRY_RUN       = False
PROJECT_ROOT  = /mnt/hpc/home/wanghongmei/lang2
CORE_DIR      = /mnt/hpc/home/wanghongmei/lang2/core
RESULTS_DIR   = /mnt/hpc/home/wanghongmei/lang2/results
时间(UTC)     = 2026-07-21T15:37:06Z
SRC_CSV       = /mnt/hpc/home/wanghongmei/lang2/results/final_candidates_paper.csv
OUT_CSV       = /mnt/hpc/home/wanghongmei/lang2/results/00_scope/all_candidates_v3_start.csv


## 1 · 环境 & provenance 快照(留痕)

In [12]:
import pandas as pd
try:
    import rdkit
    from rdkit import Chem
    RDKIT_VER = rdkit.__version__
except Exception as e:
    Chem = None
    RDKIT_VER = f"UNAVAILABLE ({e})"

env = {
    "timestamp_utc": utc_now(),
    "python": sys.version.split()[0],
    "pandas": pd.__version__,
    "rdkit": RDKIT_VER,
    "cwd": str(Path.cwd()),
}
print(json.dumps(env, indent=2, ensure_ascii=False))
assert Chem is not None, "需要 RDKit:身份键必须由与后续节点一致的 RDKit 路径生成"

{
  "timestamp_utc": "2026-07-21T15:37:11Z",
  "python": "3.10.19",
  "pandas": "2.3.3",
  "rdkit": "2025.09.4",
  "cwd": "/mnt/hpc/home/wanghongmei/lang2/core"
}


## 2 · 只读检查输入 & 记录源文件哈希(留痕)

In [13]:
assert SRC_CSV.exists(), f"找不到输入:{SRC_CSV}"
src_hash = file_sha256(SRC_CSV)
src_df   = pd.read_csv(SRC_CSV)

print("源文件         :", SRC_CSV)
print("源文件 sha256  :", src_hash)
print("源文件形状     :", src_df.shape)
print("是否含 smiles  :", "smiles" in src_df.columns)
print("唯一 smiles 数 :", src_df["smiles"].nunique())
# 关键前置证据:出身分布(重画富集图会用到)
if "is_bbs_guided" in src_df.columns:
    print("is_bbs_guided  :", src_df["is_bbs_guided"].value_counts().to_dict())
if "PySCF_ChargeNote" in src_df.columns:
    print("ChargeNote     :", src_df["PySCF_ChargeNote"].value_counts(dropna=True).head(8).to_dict())

源文件         : /mnt/hpc/home/wanghongmei/lang2/results/final_candidates_paper.csv
源文件 sha256  : 22f6399162ee09622aa59fb96269c411716f69f860ef57036c869bf2f2028221
源文件形状     : (1606, 67)
是否含 smiles  : True
唯一 smiles 数 : 1606
is_bbs_guided  : {1: 932, 0: 674}
ChargeNote     : {'neutral': 337, 'neutralized(was_charge=1)': 12, 'neutralized(was_charge=-1)': 7, 'neutralized(was_charge=2)': 3, 'neutralized(was_charge=4)': 1, 'neutralized(was_charge=3)': 1}


## 3 · 导入 core 算法(不内联)

In [14]:
# 让 notebook 找到 core/ 模块(core/ 在 PROJECT_ROOT 下)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from core import Node00_property_carryforward as pcf
print("Node00_property_carryforward version =", pcf.PROPERTY_CARRYFORWARD_VERSION)
print("KEEP 属性列数 :", len(pcf.KEEP_PROPERTY_COLS))
print("PROVENANCE    :", pcf.PROVENANCE_COLS)
print("DOCKING 黑名单:", pcf.DOCKING_BLOCKLIST)

Node00_property_carryforward version = 0.3.0
KEEP 属性列数 : 35
PROVENANCE    : ['source_runs', 'source_count', 'run_types', 'is_bbs_guided', 'primary_source']
DOCKING 黑名单: ['Broad_Spectrum_Score', 'Broad_Rank', 'Broad_Rank_Pct', 'E_SARS_CoV_2', 'E_SARS_CoV_1', 'E_MERS_CoV', 'Docking_Rank', 'Docking_Rank_Pct', 'Selection_Mode']


## 4 · 提取(核心一步)

`run_extract` 内部:RDKit canonical 化 → 去重 → 只留 KEEP+PROVENANCE →
派生 `qm_status` → 剥离并断言无 docking 列泄漏 → QC 门 → 写 manifest。
`DRY_RUN=True` 时**不落盘**,仅返回结果供核验。

In [15]:
start, manifest = pcf.run_extract(
    SRC_CSV, OUT_CSV, OUT_MANIFEST,
    pd, Chem, dry_run=DRY_RUN,
)

s = manifest["stats"]
print("唯一分子数        :", s["n_unique_molecules"])
print("去重折叠数        :", s["n_duplicate_canonical_collapsed"])
print("无法解析 SMILES   :", s["n_unparseable_smiles_dropped"])
print("携带属性列数      :", len(s["carried_property_columns"]))
print("携带出身列        :", s["carried_provenance_columns"])
print("qm_status 分布    :", s["qm_status_counts"])
print("QC 通过           :", manifest["qc_gate"]["passed"], manifest["qc_gate"]["problems"])

唯一分子数        : 1606
去重折叠数        : 0
无法解析 SMILES   : 0
携带属性列数      : 24
携带出身列        : ['source_runs', 'source_count', 'run_types', 'is_bbs_guided', 'primary_source']
qm_status 分布    : {'none': 1245, 'ok': 337, 'needs_recompute': 24}
QC 通过           : True []


## 5 · 留痕核验:输出干净性(防污染回流)

In [16]:
# 5a 断言:输出里绝无任何 docking 列
leaked = [c for c in pcf.DOCKING_BLOCKLIST if c in start.columns]
assert not leaked, f"发现 docking 列泄漏:{leaked}"
print("[OK] 输出无 docking 列泄漏")

# 5b 断言:未分类列为空(每列要么有意携带,要么有意排除)
assert s["unclassified_columns_left_out"] == [], s["unclassified_columns_left_out"]
print("[OK] 无未分类列")

# 5c 展示输出结构与样例
print("输出列:", list(start.columns))
print("输出形状:", start.shape)
show = [c for c in ["canonical_smiles","name","is_bbs_guided","primary_source",
                    "admet_QED","admet_hERG_Prob","gap_ev","qm_status"] if c in start.columns]
start[show].head(8)

[OK] 输出无 docking 列泄漏
[OK] 无未分类列
输出列: ['canonical_smiles', 'name', 'source_runs', 'source_count', 'run_types', 'is_bbs_guided', 'primary_source', 'pIC50', 'admet_QED', 'admet_Lipinski_Pass', 'admet_Violations', 'admet_MW', 'admet_LogP', 'admet_HBD', 'admet_HBA', 'admet_RotBonds', 'admet_TPSA', 'admet_hERG_Prob', 'admet_hERG_Risk', 'SA', 'esp_min', 'EHOMO_ev', 'ELUMO_ev', 'gap_ev', 'PySCF_ChargeNote', 'PySCF_Gap_eV', 'PySCF_Dipole_Debye', 'PySCF_HOMO_eV', 'PySCF_LUMO_eV', 'PySCF_Energy_Eh', 'Calc_Method', 'qm_status']
输出形状: (1606, 32)


,canonical_smiles,name,is_bbs_guided,primary_source,admet_QED,admet_hERG_Prob,gap_ev,qm_status
0,CNC(=O)c1cc(C(=O)NC(=O)c2cccc3cccnc23)cc(-c2cc...,pool_mol_001,1,11VAE_BBS_cycle1,0.388,0.436667,1.8349,ok
1,Cn1cc2cc(-c3cc(F)c(F)cc3-c3cc4[nH]c(=O)n(C(N)=...,pool_mol_002,1,11VAE_BBS_cycle1,0.411,0.483333,2.0957,ok
2,NC(=O)Cc1ccnc(-c2nc(-c3ccnc(CC(N)=O)c3)sc2-c2c...,pool_mol_003,0,01.6good200-30,0.415,0.443333,1.8934,needs_recompute
3,O=c1[nH]c(Cc2cccc(-c3nc4cc(O)c(O)c(O)c4c(=O)[n...,pool_mol_004,1,12VAE_BBS_cycle2,0.176,0.486667,2.5180,ok
4,O=c1[nH]c(-c2cccc(-c3cccc(-c4cccnc4)c3)c2)nc2c...,pool_mol_005,1,11VAE_BBS_cycle1,0.319,0.456667,2.3506,ok
5,NC(=O)Cc1ccnc(-c2cc(CC(N)=O)c(O)c(-c3ccnc(-c4c...,pool_mol_006,0,01.1更新第一层后的3*50*30,0.271,0.456667,2.4935,ok
6,NC(=O)Cc1ccnc(-c2ccnc(-c3cc(-c4ccc(CC(N)=O)cc4...,pool_mol_007,0,01.1更新第一层后的3*50*30,0.267,0.446667,2.4956,ok
7,Cc1cncc(Nc2nc(=O)n(-c3cncc(C)c3)c(=O)n2Cc2cc(F...,pool_mol_008,1,14VAE_BBS_cycle4,0.466,0.460444,2.3937,ok


## 6 · 落盘(仅 `DRY_RUN=False`)+ 记录输出哈希

In [17]:
if DRY_RUN:
    print("DRY_RUN=True:未写盘。核验以上无误后,将顶部 DRY_RUN 改为 False 重跑本 notebook。")
else:
    assert OUT_CSV.exists() and OUT_MANIFEST.exists(), "落盘失败"
    out_hash = file_sha256(OUT_CSV)
    # 源哈希 + 输出哈希写入溯源文件,形成完整链路留痕
    trace = {
        "created_at_utc": utc_now(),
        "source_csv": str(SRC_CSV),
        "source_sha256": src_hash,
        "output_csv": str(OUT_CSV),
        "output_sha256": out_hash,
        "module_version": pcf.PROPERTY_CARRYFORWARD_VERSION,
        "n_unique_molecules": s["n_unique_molecules"],
        "qm_status_counts": s["qm_status_counts"],
    }
    OUT_TRACE.write_text(json.dumps(trace, indent=2, ensure_ascii=False), encoding="utf-8")
    print("已写:", OUT_CSV)
    print("输出 sha256:", out_hash)
    print("溯源:", OUT_TRACE)

已写: /mnt/hpc/home/wanghongmei/lang2/results/00_scope/all_candidates_v3_start.csv
输出 sha256: 64c1d88ad3986ef4de9a775af9a7c0b24b965170bc3ec643469d3045f2196e78
溯源: /mnt/hpc/home/wanghongmei/lang2/results/00_scope/all_candidates_v3_start.trace.json


## 7 · 节点状态板 & 下一步

- **产物**:`results/00_scope/all_candidates_v3_start.csv`(V3 出发文件,1606×32)
- **本步固定**:身份 *范围*(canonical SMILES) + 出身 + 干净 ADMET/QM(带 `qm_status`)
- **本步未做**:权威 *对象*(质子化/电荷微观态、3D 主结构)→ 交给 **Node 01–02**
- **待办**:`qm_status == needs_recompute` 的 24 个,Node 01 定电荷后重算 QM


In [21]:
# ══════════════════════════════════════════════════════════════════
#  Node 00 追加单元 · pool_class 回填 + 碎片溯源  (v2, 含运行前防护)
#  贴到 Node00_make_v3_start.ipynb 末尾运行。仅追加 3 列,不重建、不删行。
# ══════════════════════════════════════════════════════════════════
from pathlib import Path
import sys, pandas as pd
from rdkit import Chem
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from core import node00_pool_class_backfill as PC

DRY_RUN = False     # 确认无误后改 False 落盘

IN_CSV   = RESULTS_DIR / "00_scope" / "all_candidates_v3_start.csv"
OUT_CSV  = RESULTS_DIR / "00_scope" / "all_candidates_v3_start.csv"   # 原地回填
MANIFEST = RESULTS_DIR / "00_scope" / "pool_class_backfill.manifest.json"

df = pd.read_csv(IN_CSV)

# ── 防护1(审计要求):SMILES 列名可能是 smiles 而非 canonical_smiles ──
smiles_col = PC.SMILES_COL
if smiles_col not in df.columns:
    for alt in ("smiles", "SMILES", "Smiles"):
        if alt in df.columns:
            smiles_col = alt
            print(f"[防护] 未见 {PC.SMILES_COL},改用实际列 '{alt}'")
            break
assert smiles_col in df.columns, \
    f"缺 SMILES 列;实际列: {list(df.columns)}"
assert PC.NAME_COL in df.columns, \
    f"缺 {PC.NAME_COL} 列;实际列: {list(df.columns)}"
print(f"读入 {len(df)} 行 (Node00 输入,~1606) | 版本 {PC.BACKFILL_VERSION} "
      f"| 阈值 heavy<{PC.HEAVY_ATOM_MIN} | SMILES列='{smiles_col}'")

# ── 活2:回填 pool_class(仅依化学)──
annotated, stats = PC.backfill(df, Chem, pd, smiles_col=smiles_col)
print("\npool_class 分布:", stats["pool_class_counts"])
print(f"  candidate {stats['n_candidate']} | fragment {stats['n_fragment']} "
      f"| implausible_valence {stats['n_implausible_valence']}")
# 审计预期:implausible_valence 预期为 0(RDKit 接受 [Cl+2]/[SH](=O)=O,按尺寸落 fragment)
if stats["n_implausible_valence"] == 0:
    print("  (预期:implausible_valence=0 正确 —— RDKit 严格 sanitize 接受这些异常写法,")
    print("   它们按 4 重原子落入 fragment,聚类结果一致)")
print(f"  fragment 数 {stats['n_fragment']} (审计预期 ~70–76;76 是 built-pool 代理值,"
      f"Node00 输入上略低属正常)")

# ── 活1:碎片按来源溯源(仅读 source 列,供 SI)──
trace = PC.trace_fragment_sources(annotated, source_col="primary_source")
if "error" not in trace:
    print("\n碎片/异常 按来源分布:")
    for src, n in sorted(trace["flagged_by_source"].items(), key=lambda x: -x[1]):
        print(f"    {src}: {n}")
    print(f"  → {trace['note']}")
else:
    print("\n[溯源] 跳过:", trace["error"])

# ── 落盘(fail-safe:默认 dry_run)──
manifest = PC.write_outputs(annotated, stats, OUT_CSV, MANIFEST, IN_CSV, dry_run=DRY_RUN)
if DRY_RUN:
    print("\n[DRY_RUN] 未写盘。核验计数无误后设 DRY_RUN=False 重跑本单元。")
else:
    print(f"\n已回填: {OUT_CSV}\nmanifest: {MANIFEST}")
    print("下游用法: 全部对接;聚类时仅取 pool_class=='candidate'")


读入 1606 行 (Node00 输入,~1606) | 版本 0.1.0 | 阈值 heavy<12 | SMILES列='canonical_smiles'

pool_class 分布: {'candidate': 1530, 'fragment': 76}
  candidate 1530 | fragment 76 | implausible_valence 0
  (预期:implausible_valence=0 正确 —— RDKit 严格 sanitize 接受这些异常写法,
   它们按 4 重原子落入 fragment,聚类结果一致)
  fragment 数 76 (审计预期 ~70–76;76 是 built-pool 代理值,Node00 输入上略低属正常)

碎片/异常 按来源分布:
    07VAE_2+U-3_200_50: 40
    02VAE.1重训后3*50*30: 19
    09VAE_2+U-3_100_30: 5
    08VAE_2+U-3_100_30: 5
    01VAE.0后的3*50*30: 4
    03VAE_2+U-3_100_30: 2
    01.5good-3*50*30: 1
  → fragments are spread across early under-trained VAE runs, not a single contaminated file

已回填: /mnt/hpc/home/wanghongmei/lang2/results/00_scope/all_candidates_v3_start.csv
manifest: /mnt/hpc/home/wanghongmei/lang2/results/00_scope/pool_class_backfill.manifest.json
下游用法: 全部对接;聚类时仅取 pool_class=='candidate'
